In [14]:
print("Test")

Test


In [15]:
# Imports
import sys
import pathlib

# Add the project's root directory to the Python path
sys.path.append(pathlib.Path("../").resolve().as_posix())

# Configurations
seed = 42

# Paths
DATA_DIR = pathlib.Path("../data/")
ENC2017_ROOT = DATA_DIR / "enc2017"
UD_ET_EDT_ROOT = DATA_DIR / "ud_et_edt"
HOMONYMS_ROOT = DATA_DIR / "homonymous_word_forms"

ENC2017_DIRS = {
    "processed": ENC2017_ROOT / "processed",
    "raw": ENC2017_ROOT / "raw",
}

UD_ET_EDT_DIRS = {
    "processed": UD_ET_EDT_ROOT / "processed",
    "raw": UD_ET_EDT_ROOT / "raw",
}

HOMONYMS_DIRS = {
    "processed": HOMONYMS_ROOT / "processed",
    "annotations": HOMONYMS_ROOT / "annotations",
}

OUTPUT_DIR = pathlib.Path("../outputs/")

MODEL_DIR = pathlib.Path("../models/")

In [39]:
import json
import estnltk
import tqdm
import pandas as pd
from estnltk.default_resolver import make_resolver

from scripts.model import initialize_model
from scripts.model.bert_morph_tagger import BertMorphTagger

In [40]:
conflicts_dir = DATA_DIR / "conflicts"
bmt = BertMorphTagger(model_location=str(MODEL_DIR / "NER_mudel_v2"))

In [41]:
# Open JSON file
with open(
    conflicts_dir / "raw" / "nsubj_conflicts_dataset_15022026.json",
    "r",
    encoding="utf-8",
) as f:
    # Load JSON data
    conflicts_data = json.load(f)

In [ ]:
resolver = make_resolver()

In [52]:
for i, row in tqdm.tqdm(enumerate(conflicts_data), total=len(conflicts_data)):
    sentence = row["sentence"]
    word = row["word_form"]
    word_span = row["span"]
    # print(f"Sentence: {sentence}")
    # print(f"Word: {word}, Span: {word_span}")
    text_obj = estnltk.Text(sentence)
    text_obj.tag_layer(["sentences", "words"])
    bmt.tag(text_obj)
    forms = []
    for ma in text_obj.bert_morph_tagging:
        if ma.start == word_span[0] and ma.end == word_span[1]:
            # Extract the form and pos
            for form in ma.form:
                forms.append(form)
    row["bert_morph_v2_form"] = forms

 78%|███████▊  | 7506/9634 [05:47<01:37, 21.84it/s]E:\Git_projects/EstNLTK/EstNLTK_model_training\scripts\model\bert_tokens_to_words_rewriter.py:228: UserWarning: (!) No matching words span for bert token Span('', [{'bert_tokens': '▁»', 'form': '', 'partofspeech': 'Z', 'probability': 0.99999}]).
  warnings.warn(
100%|██████████| 9634/9634 [07:24<00:00, 21.67it/s]


In [ ]:
# Save the updated data back to a new JSON file
with open(
    conflicts_dir
    / "processed"
    / "nsubj_conflicts_dataset_15022026_with_Bert_Morph_v2.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(conflicts_data, f, ensure_ascii=False, indent=2)

In [57]:
display(conflicts_data[0])

{'sentence_id': 12410790,
 'sentence': 'Kristina Shmiguni ahistas Holmenkollenis 30 km klassikatehnikasõidu lõpukilomeetritel kurnatus , ent viiendast kohast maailma karikasarja liidrirüü hoidmiseks piisas .',
 'word_form': 'Kristina',
 'span': [0, 8],
 'current_deprel': 'nsubj',
 'current_case': 'gen',
 'current_analysis': 'gen,prop,sg',
 'bert_morph_v2_form': ['sg n']}

In [61]:
# Map the given forms to Vabamorf format
map_forms = {
    "n": "nom",
    "g": "gen",
    "p": "par",
    "adt": "ill",
    "ill": "ill",
    "in": "ine",
    "el": "ela",
    "all": "all",
    "ad": "ade",
    "abl": "abl",
    "tr": "tra",
    "ter": "trm",
    "es": "ess",
    "ab": "abe",
    "kom": "com",
}

In [82]:
def parse_bmt_form(form):
    # If the form is None or empty, return None
    if form is None or form == [""]:
        return None, None
    form_data = form[0].split(" ")
    # If the form is related to a verb
    if len(form_data) == 1:
        word_form = form_data[0]
        return word_form, None
    # Split the form into count and form
    else:
        count = form_data[0]
        word_form = form_data[1]
        # Map the form to Vabamorf format
        mapped_form = map_forms.get(word_form, word_form)
        return word_form, mapped_form

In [80]:
conflicts_data[107]

{'sentence_id': 15887988,
 'sentence': 'Mis ma oskan sul kosta , proovi jah uuemaid draivereid , proovi mälu CAS 2 kui ei aita , siis oled stuck ...',
 'word_form': 'proovi',
 'span': [57, 63],
 'current_deprel': 'nsubj',
 'current_case': 'gen',
 'current_analysis': 'com,gen,sg',
 'bert_morph_v2_form': ['o']}

In [99]:
# Count:
# - the number of matching forms
# - the number of non-matching forms
# - the number of no predicted forms (bert morph v2 form is None)
# - the number of non-matching forms where the Bert Morph v2 form is in ["n", "p"]
matching_forms_count = 0
non_matching_forms_count = 0
no_predicted_forms_count = 0
non_matching_forms = {}
for i, row in tqdm.tqdm(enumerate(conflicts_data), total=len(conflicts_data)):
    sentence = row["sentence"]
    word = row["word_form"]
    word_span = row["span"]
    bert_Morph_v2_form = row["bert_morph_v2_form"]
    bmt_form, parsed_form = parse_bmt_form(bert_Morph_v2_form)
    vabamorf_form = row["current_case"]
    # print(f"Sentence: {sentence}")
    # print(f"Word: {word}, Span: {word_span}")
    # print(f"Bert Morph v2 form: {bert_Morph_v2_form}")
    # print(f"Vabamorf form: {vabamorf_form}")
    if parsed_form is None:
        no_predicted_forms_count += 1
    elif parsed_form == vabamorf_form:
        matching_forms_count += 1
    else:
        non_matching_forms_count += 1
        non_matching_forms[bmt_form] = non_matching_forms.get(bmt_form, 0) + 1

100%|██████████| 9634/9634 [00:00<00:00, 921356.33it/s]


In [106]:
non_matching_forms_with_np_count = non_matching_forms.get(
    "n", 0
) + non_matching_forms.get("p", 0)

print(
    f"Total forms count: {len(conflicts_data)} (100.00%) (Percentages are calculated based on the total number of forms in the dataset)"
)
print(
    f"Matching forms count: {matching_forms_count} ({matching_forms_count / len(conflicts_data) * 100:.2f}%)"
)
print(
    f"Non-matching forms count: {non_matching_forms_count} ({non_matching_forms_count / len(conflicts_data) * 100:.2f}%)"
)
print(
    f"No predicted forms count: {no_predicted_forms_count} ({no_predicted_forms_count / len(conflicts_data) * 100:.2f}%)"
)
print(
    f"Non-matching forms where Bert Morph v2 form is in ['n', 'p'] count: {non_matching_forms_with_np_count} ({non_matching_forms_with_np_count / len(conflicts_data) * 100:.2f}%)"
)
print(
    f"Non-matching forms where Bert Morph v2 form is 'g' count: {non_matching_forms['g']} ({non_matching_forms['g'] / len(conflicts_data) * 100:.2f}%)"
)
print("All non-matching forms counts:")
for form, count in non_matching_forms.items():
    print(f"  {form}: {count} ({count / len(conflicts_data) * 100:.2f}%)")

Total forms count: 9634 (100.00%) (Percentages are calculated based on the total number of forms in the dataset)
Matching forms count: 1551 (16.10%)
Non-matching forms count: 7712 (80.05%)
No predicted forms count: 371 (3.85%)
Non-matching forms where Bert Morph v2 form is in ['n', 'p'] count: 6931 (71.94%)
Non-matching forms where Bert Morph v2 form is 'g' count: 63 (0.65%)
All non-matching forms counts:
  n: 5202 (54.00%)
  p: 1729 (17.95%)
  all: 487 (5.06%)
  tr: 158 (1.64%)
  el: 25 (0.26%)
  es: 4 (0.04%)
  g: 63 (0.65%)
  ad: 13 (0.13%)
  ks: 1 (0.01%)
  ill: 6 (0.06%)
  in: 4 (0.04%)
  o: 7 (0.07%)
  kom: 7 (0.07%)
  ter: 4 (0.04%)
  ge: 1 (0.01%)
  abl: 1 (0.01%)
